# Phase A: Mask Validation

**One-time diagnostic** — confirm padding/masking fixes are working end-to-end.
Run 1 epoch of Phase 1 only with `debug_masks=True`.

**What to check:**
- No assertion errors
- `[MASK]` lines: ASVspoof source ~0-5% padding, ITW target higher variance
- `[MASK-POOL]` lines: frame_mask shape matches hidden state dims
- No warnings about fully-padded samples

**After validation:** Set `debug_masks=False` for all subsequent phases.


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy glada package from Drive to local SSD (faster imports)
import shutil, os, sys
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code"
LOCAL_PKG = '/content/glada_code'
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)
sys.path.insert(0, LOCAL_PKG)
print(f'Package copied to {LOCAL_PKG}')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected')


## 2. Dataset Extraction


In [2]:
import os
import shutil
import tarfile
import zipfile

DRIVE_DATASETS = '/content/drive/MyDrive/Jakob/Datasets'
LOCAL_DATA = '/content/data'


def extract_one_dataset(drive_archive, extract_to, verify_path, name):
    """Copy archive from Drive -> runtime SSD, extract, delete archive.

    Processes one dataset at a time so the runtime SSD never holds
    more than one compressed archive alongside its decompressed data.
    """
    # Skip if already extracted (e.g., runtime reconnect without wipe)
    if os.path.isdir(verify_path) and os.listdir(verify_path):
        print(f'  [{name}] Already extracted at {verify_path}')
        return

    archive_name = os.path.basename(drive_archive)
    local_archive = f'/content/{archive_name}'

    # Step 1: Copy archive to runtime SSD
    print(f'  [{name}] Copying {archive_name} to runtime SSD...')
    shutil.copy2(drive_archive, local_archive)
    size_gb = os.path.getsize(local_archive) / (1024 ** 3)
    print(f'  [{name}] Copied ({size_gb:.2f} GB)')

    # Step 2: Extract locally (faster than extracting from Drive over FUSE)
    os.makedirs(extract_to, exist_ok=True)
    print(f'  [{name}] Extracting to {extract_to}...')
    if archive_name.endswith('.zip'):
        with zipfile.ZipFile(local_archive, 'r') as zf:
            zf.extractall(extract_to)
    elif archive_name.endswith('.tar.gz') or archive_name.endswith('.tgz'):
        with tarfile.open(local_archive, 'r:gz') as tf:
            tf.extractall(extract_to)
    elif archive_name.endswith('.tar'):
        with tarfile.open(local_archive, 'r:') as tf:
            tf.extractall(extract_to)
    else:
        raise ValueError(f'Unknown archive format: {archive_name}')

    # Step 3: Delete archive from SSD (free space before next dataset)
    os.remove(local_archive)
    print(f'  [{name}] Done. Archive deleted from SSD.\n')


# --- Process each dataset one at a time ---
datasets = [
    {
        'name': 'ASVspoof 2019 LA',
        'drive_archive': f'{DRIVE_DATASETS}/LA.tar',
        'extract_to': f'{LOCAL_DATA}/asvspoof',
        'verify_path': f'{LOCAL_DATA}/asvspoof/LA',
    },
    {
        'name': 'In-the-Wild',
        'drive_archive': f'{DRIVE_DATASETS}/release_in_the_wild.tar',
        'extract_to': f'{LOCAL_DATA}/itw',
        'verify_path': f'{LOCAL_DATA}/itw/release_in_the_wild',
    },
    {
        'name': 'MUSAN',
        'drive_archive': f'{DRIVE_DATASETS}/musan.tar.gz',
        'extract_to': LOCAL_DATA,
        'verify_path': f'{LOCAL_DATA}/musan',
    },
    {
        'name': 'RIRS_NOISES',
        'drive_archive': f'{DRIVE_DATASETS}/rirs_noises.zip',
        'extract_to': LOCAL_DATA,
        'verify_path': f'{LOCAL_DATA}/RIRS_NOISES',
    },
]

print('Extracting datasets (one at a time to conserve SSD space):\n')
for ds in datasets:
    assert os.path.exists(ds['drive_archive']), \
        f"Archive not found on Drive: {ds['drive_archive']}"
    extract_one_dataset(**ds)

# --- Define paths used by later cells ---
MUSAN_PATH = '/content/data/musan'
RIR_PATH = '/content/data/RIRS_NOISES/simulated_rirs'

ASV_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO = '/content/data/itw/release_in_the_wild'

# Verify all critical paths exist
for p in [ASV_PROTO, ASV_AUDIO, ITW_META, MUSAN_PATH, RIR_PATH]:
    assert os.path.exists(p), f'Missing: {p}'

# Confirm no leftover archives on runtime SSD
leftover = [f for f in os.listdir('/content/')
            if f.endswith(('.tar', '.tar.gz', '.tgz', '.zip'))
            and os.path.isfile(f'/content/{f}')]
assert not leftover, f'Leftover archives on SSD (should be deleted): {leftover}'

print('All datasets extracted and verified. No compressed archives on SSD.')

Extracting datasets (one at a time to conserve SSD space):

  [ASVspoof 2019 LA] Copying LA.tar to runtime SSD...
  [ASVspoof 2019 LA] Copied (7.22 GB)
  [ASVspoof 2019 LA] Extracting to /content/data/asvspoof...


/tmp/ipykernel_3044/3258827901.py:41: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(extract_to)


  [ASVspoof 2019 LA] Done. Archive deleted from SSD.

  [In-the-Wild] Copying release_in_the_wild.tar to runtime SSD...
  [In-the-Wild] Copied (8.15 GB)
  [In-the-Wild] Extracting to /content/data/itw...
  [In-the-Wild] Done. Archive deleted from SSD.

  [MUSAN] Copying musan.tar.gz to runtime SSD...
  [MUSAN] Copied (10.32 GB)
  [MUSAN] Extracting to /content/data...


/tmp/ipykernel_3044/3258827901.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(extract_to)


  [MUSAN] Done. Archive deleted from SSD.

  [RIRS_NOISES] Copying rirs_noises.zip to runtime SSD...
  [RIRS_NOISES] Copied (1.22 GB)
  [RIRS_NOISES] Extracting to /content/data...
  [RIRS_NOISES] Done. Archive deleted from SSD.

All datasets extracted and verified. No compressed archives on SSD.


## 3. Imports


In [ ]:
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.datasets import (
    ASVspoofDatasetWithSpeaker,
    ASVspoofEvalDataset,
    InTheWildDataset,
    create_dataloaders,
)
from glada.augmentation import CombinedAugmenter
from glada.trainers import GLADATrainer

print('All imports successful.')


## 4. Mask Validation Config


In [ ]:
config = GLADAConfig(
    # --- Mask debug ---
    debug_masks=True,
    debug_mask_log_batches=5,

    # --- Minimal training (just enough to validate masks) ---
    phase1_epochs=1,
    phase2_epochs=0,

    # --- Standard settings ---
    backbone='wavlm-large',
    batch_size=24,
    amp_dtype='float16',
    num_workers=0,
    max_audio_len=64000,
    seed=42,
    experiment_name='phase_a_mask_validation',
)

set_seed(config.seed)
print(f'Config: debug_masks={config.debug_masks}, '
      f'phase1={config.phase1_epochs}, phase2={config.phase2_epochs}')


## 5. Load Datasets


In [ ]:
print('Loading datasets...')

# Build augmentation pipeline
augmenter = CombinedAugmenter(config)
print(f'Augmenter: {augmenter}')

# Source: ASVspoof 2019 LA (train split)
source_ds = ASVspoofDatasetWithSpeaker(
    protocol_path=ASV_PROTO,
    audio_dir=ASV_AUDIO,
    config=config,
    augmenter=augmenter,
)

# Target: In-The-Wild (unlabeled for domain adaptation)
target_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=False,
)

# Eval: In-The-Wild (labeled for EER computation)
eval_ds = InTheWildDataset(
    meta_csv_path=ITW_META,
    audio_dir=ITW_AUDIO,
    config=config,
    use_labels=True,
)

loaders = create_dataloaders(config, source_ds, target_ds, eval_ds)
source_loader = loaders['source']
target_loader = loaders['target']
eval_loader = loaders['eval']

print(f'Source: {len(source_ds):,} samples, Target: {len(target_ds):,}, '
      f'Eval: {len(eval_ds):,}')


## 6. Run Mask Validation

Watch the output for `[MASK]` and `[MASK-POOL]` log lines.


In [ ]:
trainer = GLADATrainer(
    config=config,
    source_loader=source_loader,
    target_loader=target_loader,
    eval_loader=eval_loader,
)

results = trainer.train()

print('\n' + '=' * 60)
print('MASK VALIDATION COMPLETE')
print('=' * 60)
print(f'Phase 1 EER: {results.get("best_eer", "N/A")}')
print('\nIf no assertion errors appeared above and [MASK]/[MASK-POOL] stats')
print('look reasonable, masking is working correctly.')
print('Set debug_masks=False for all subsequent phases.')
